# 00 · Data check

What's in the database, how much of it there is, and whether the sources are healthy. Run it top to bottom. Clear the outputs before committing (research/README.md).

In [ ]:
from topnews import db

engine = db.engine()  # read-only; RESEARCH_DATABASE_URL from .env.local
db.table_counts(engine)

## Source health over the last 7 days

The same numbers as TASKS.md › Gate 1: each source's runs and the share that succeeded.

In [ ]:
runs = db.fetch_runs(engine, days=7)
runs = runs[runs.source_id != "heartbeat"]
health = (
    runs.assign(ok=runs.status.eq("ok"))
    .groupby("source_id")
    .agg(runs=("ok", "size"), success_pct=("ok", lambda s: round(100 * s.mean(), 1)))
    .sort_values("success_pct")
)
health

## When do new items appear? (a first look at RQ5)

For each source, the hour (UTC) at which each item was first seen in its list. TikTok refreshes daily and Instagram every 6 hours, so their hourly patterns aren't meaningful.

In [ ]:
items = db.trend_items(engine, days=7)
items["key"] = items.title.str.lower().str.strip()
first = items.sort_values("fetched_at").drop_duplicates(["source_id", "key"])
first.assign(hour_utc=first.fetched_at.dt.hour).pivot_table(
    index="hour_utc", columns="source_id", values="key", aggfunc="count", fill_value=0
)

## Topic snapshots

The main dataset. It starts filling once phase 2 is merged. Keep an analysis to one `algo_version`.

In [ ]:
snaps = db.snapshots(engine)
if snaps.empty:
    print("No snapshots yet: they start when phase 2 is merged.")
else:
    print(f"{len(snaps):,} rows, {snaps.topic_id.nunique():,} topics, {snaps.taken_at.min()} to {snaps.taken_at.max()}")
    display(snaps.algo_version.value_counts())